In [1]:
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever, BM25Retriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import DatabricksVectorSearch

C:\Users\soldesk\AppData\Local\Temp\ipykernel_5964\1508242155.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


ImportError: cannot import name 'DatabricksVectorSearch' from 'langchain_community.vectorstores' (c:\source\ollama\.venv\Lib\site-packages\langchain_community\vectorstores\__init__.py)

In [19]:
# .env 내용 가져오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    max_tokens=2000,
)

ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params = {
        "temperature" : 0
    }
)

qwen_llm = ChatOllama(model="qwen3.5:4b", temperature=0)
exaone_llm = ChatOllama(model="exaone3.5:2.4b", temperature=0)

### RAG 심화




In [20]:
# pdf => chunks 반환 함수

def create_chunks_from_pdf(pdf_path, chunk_size=500, chunk_overlap=50):
    loader = PyPDFLoader(pdf_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(loader.load())

    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]

    return chunks

def create_vectorstore(chunks, embeddings, collection_name, persis_directory='./db/chroma_db'):
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persis_directory,
        collection_name=collection_name
    )

def create_retriever(vectorstore ,search_type="similarirty",k=3,fetch_k=20,lambda_mult=0.5):
    kwargs = {"k":k}

    if search_type=="mmr":
        kwargs['fetch_k'] = fetch_k
        kwargs['lambda_mult'] = lambda_mult

    return vectorstore.as_retriever(search_type=search_type, search_kwargs=kwargs)

def print_retrieved_docs(title, retriever, query):
    docs = retriever.invoke(query)

    print("\n" + "="*50)
    print(title)
    print("\n" + "="*50)    

    for i, doc in enumerate(docs):
        print(f"\n [chunk {i}]")
        print(doc.page_content)
        print(f"\nPage : {doc.metadata.get("page")}")


In [21]:
chunks1 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 1000, 100)
chunks2 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 300, 30)

print(f"분할된 청크 수 : {len(chunks1)}")
print(f"분할된 청크 수 : {len(chunks2)}")

분할된 청크 수 : 118
분할된 청크 수 : 378


In [22]:
vectorstore1 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_ollama3")
vectorstore2 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_ollama4")

ollama1_retriever = create_retriever(vectorstore1)
ollama2_retriever = create_retriever(vectorstore2)

query = 'where can i use chatGPT?'

print_retrieved_docs('ollama embedding', ollama1_retriever, query)
print_retrieved_docs('ollama embedding', ollama2_retriever, query)

ValidationError: 1 validation error for VectorStoreRetriever
  Value error, search_type of similarirty not allowed. Valid values are: ('similarity', 'similarity_score_threshold', 'mmr') [type=value_error, input_value={'vectorstore': <langchai...earch_kwargs': {'k': 3}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

### 2. MMR (Maximal Marginal Relevance) Retriever
 - 관련성(Relevace)과 다양성(Diversity) 고려
 - 법률 문서, 기술 매뉴얼 유사 내용이 반복되는 문서에서 효과적이다.

In [25]:
chunks1 = create_chunks_from_pdf(
    "./data/2026 상 삼성전자 DX부문 직무기술서.pdf",
    1000,
    100,
)

vectorstore1 = create_vectorstore(
    chunks1, ollama_embedding, collection_name="samsung_watson1", persist_directory="./db/samsung_watson_db"
)

mmr_retriever = create_retriever(vectorstore1, search_type="mmr", k=5)
similarity_retriever = create_retriever(vectorstore1, k=5)

query = '마케팅 - 제품/서비스 마케팅 포지션은?'

print_retrieved_docs('MMR', mmr_retriever, query)
print_retrieved_docs('similarity', similarity_retriever, query)


TypeError: create_vectorstore() got an unexpected keyword argument 'persist_directory'

# 3. selfquery retriever
- 자연어 질문을 분석하여 시멘틱 검색쿼리와 메타데티어 필터를 llm이 자동 생성하게 하는 고급 retriever
- 질문 : 2023 년 이후 기약금액이 1억 이상인 계약 찾아줘 => LLM
    시멘틱 검색 쿼리 : 계약
    filter : {year >= 2023, 계약금액 >= 100000}

In [26]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata={
            "year":2025,
            "department":"marketing"
        }
    ),
        Document(
        page_content="삼성전자 회계 직무 입니다.",
        metadata={
            "year":2024,
            "department":"accounting"
        }
    ),
        Document(
        page_content="삼성전자 백엔드 직무입니다.",
        metadata={
            "year":2023,
            "department":"backend"
        }
    )
]

metadata_field_info = [
    AttributeInfo(
        name="year", description="문서작성 연도", type="integer"),
    AttributeInfo(
        name="department", description="직무 부서", type="string"
    )
]


document_content_description = "회사 내부 문서 및 직무 자료"

In [27]:
vectorstore1 = create_vectorstore(
    docs,
    watson_embedding,
    collection_name="selfquery",
    persis_directory='./db/watson_chroma'
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore1,
    metadata_field_info=metadata_field_info,
    document_contents=document_content_description,
    verbose=True,
    enable_limit=True,
    structed_query_translator=ChromaTranslator()
)


ImportError: cannot import name 'DatabricksVectorSearch' from 'langchain_community.vectorstores' (c:\source\ollama\.venv\Lib\site-packages\langchain_community\vectorstores\__init__.py)